# RPO (legacy, step-by-step)

The original hand-chained plan: each leg is propagated separately and stitched together with `ts_0_to_N` arrays. Superseded by the `RpoTimeline` version in `rpo.ipynb`; kept for reference.

In [480]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import numpy as np
import plotly.graph_objects as go
from cesium.constants import (
    DAY_TO_SEC,
    DEG_TO_RAD,
    KM_TO_M,
    M_TO_KM,
    RAD_TO_DEG,
    SEC_TO_DAY,
    SEC_TO_HOUR,
    SEC_TO_MIN,
)
from cesium.guidance.coplanar import guid_coaxial_hohmann, guid_phasing_same_orbit
from cesium.guidance.noncoplanar import guid_combined_i_raan_change, node_line_finder
from cesium.math.base import unit
from cesium.math.rotations import wrap_to_2pi, wrap_to_pi
from cesium.orbital.coes import COES, orbit_sweep
from cesium.orbital.hcw import hcw_pci_to_rsw
from cesium.physics.dynamics import propagate_two_body
from cesium.plotting import plot_3d, plot_separation
from cesium.world.constants import J2_MARS, MU_MARS, R_MARS
from numpy.linalg import norm
from tqdm import tqdm

In [ ]:
def print_diffs(coes_t: COES, coes_c1: COES, coes_c2: COES = None, mu: float = MU_MARS):
    """Print target and chaser COES and their differences (plus a burn's change if `coes_c2`)."""
    coes_t.print("target            ", mu)
    coes_c1.print("chaser before     ", mu)
    coes_c1.delta(coes_t).print("before - target   ", mu)
    if coes_c2 is not None:
        print()
        coes_c2.print("chaser after      ", mu)
        coes_c2.delta(coes_c1).print("change            ", mu)
        coes_c2.delta(coes_t).print("after - target    ", mu)

## Setup
Assume within 5 degrees inclination, raan, 0.05 e, 100km a

In [482]:
# Target orbit (same as arrival.ipynb), MCI, circular
PARKING_ORBIT_ALT_M = 300e3  # m
PARKING_ORBIT_RAAN_DEG = 15
PARKING_ORBIT_I_DEG = 45
PARKING_ORBIT_TA_DEG = 13

TARGET_0 = COES(
    a=R_MARS + PARKING_ORBIT_ALT_M,
    e=0.0,
    i=PARKING_ORBIT_I_DEG * DEG_TO_RAD,
    raan=PARKING_ORBIT_RAAN_DEG * DEG_TO_RAD,
    aop=0.0,
    ta=PARKING_ORBIT_TA_DEG * DEG_TO_RAD,
)
a_t, e_t, i_t, raan_t, aop_t, ta_t = TARGET_0

dt = 1  # sec

In [694]:
# Chaser = target + seeded uniform dispersion
# SEED = 525600
SEED = 3214359876
# SEED = 13
ANGLE_DISP_DEG = 3.0  # i, raan, aop
A_DISP_M = 100 * KM_TO_M
E_MAX = 0.01  # e >= 0, so draw from [0, E_MAX]
J2 = 0
INITIAL_TA_OFFSET_DEG = 57

# CRAZY
# ANGLE_DISP_DEG = 50.0  # i, raan, aop
# A_DISP_M = 500 * KM_TO_M
J2 = J2_MARS

rng = np.random.default_rng(SEED)
rel_coes = np.array(
    [
        rng.uniform(-A_DISP_M, A_DISP_M),
        rng.uniform(0.0, E_MAX),
        *rng.uniform(-ANGLE_DISP_DEG, ANGLE_DISP_DEG, 3) * DEG_TO_RAD,
        INITIAL_TA_OFFSET_DEG * DEG_TO_RAD,  # ta: phasing left to the RPO problem
    ]
)
CHASER_0 = COES.from_array(TARGET_0.elements() + rel_coes)
a_c, e_c, i_c, raan_c, aop_c, ta_c = CHASER_0

TARGET_0.print("initial target coes")
CHASER_0.print("initial chaser coes")

# mean motion
n_c0 = np.sqrt(MU_MARS / a_c**3)
n_t0 = np.sqrt(MU_MARS / a_t**3)

initial target coes: a = 3696.000 km, e = 0.00000, i = 45.000 deg, raan = 15.000 deg, aop = 0.000 deg, ta = 13.000 deg
initial chaser coes: a = 3702.270 km, e = 0.00621, i = 44.231 deg, raan = 17.855 deg, aop = -2.894 deg, ta = 70.000 deg


In [695]:
# Propagate both for a few target periods
N_ORBITS = 3
period_s = 2 * np.pi * np.sqrt(a_t**3 / MU_MARS)
t_s = np.linspace(0, N_ORBITS * period_s, 3000)

t0 = 0
rv_chaser_mci = propagate_two_body(
    CHASER_0.to_rv(MU_MARS), t_s, mu=MU_MARS, j2=J2, r_cb=R_MARS
)
rv_target_mci = propagate_two_body(
    TARGET_0.to_rv(MU_MARS), t_s, mu=MU_MARS, j2=J2, r_cb=R_MARS
)

## Planning
- long way
  - raan and i
  - circularize
  - phasing
- short way
  - minimum combined plane change (but figure that out)

**Timings**
- *Time 0:* Start of trajectory
- *Time 1:* (Burn 1) RAAN/i burn at line of nodes
- *Time 2:* (Burn 2) Periapse
- *Time 3:* (Burn 3) Circularization
- *Time 4:* (Burn 4) Phasing 1
- *Time 5:* (Burn 5) Phasing 2

**Larger plan**
1. Line of nodes
2. Find time until that node
3. Burn at `t1` at that node in specified direction for `i` and `raan` change
4. Recalculate new coes

### From 0 -> 1

In [ ]:
# Burn length
n_hat, d_ta, h_chaser, h_target = node_line_finder(
    CHASER_0.to_rv(MU_MARS), TARGET_0.to_rv(MU_MARS)
)

CHASER_1_exp, dt_0_to_1 = orbit_sweep(CHASER_0, d_ta, MU_MARS)
# TARGET_1 = pkepler_coes(TARGET_0, dt_0_to_1, MU_MARS)

# Actually propagating
print(f"Coast time: {dt_0_to_1 * SEC_TO_MIN:.5f} min")
t1 = t0 + dt_0_to_1
ts_0_to_1 = np.linspace(t0, t1, 3000)
rv_c_0_to_1 = propagate_two_body(
    CHASER_0.to_rv(MU_MARS), ts_0_to_1, mu=MU_MARS, j2=J2, r_cb=R_MARS
)
rv_t_0_to_1 = propagate_two_body(
    TARGET_0.to_rv(MU_MARS), ts_0_to_1, mu=MU_MARS, j2=J2, r_cb=R_MARS
)

# Turning back into the COES
CHASER_1 = COES.from_rv(rv_c_0_to_1[-1], MU_MARS)
TARGET_1 = COES.from_rv(rv_t_0_to_1[-1], MU_MARS)

### Time 1:  Burn 1

In [698]:
# Burn
dv_i_raan = guid_combined_i_raan_change(
    CHASER_1.i,
    TARGET_1.i,
    TARGET_1.raan - CHASER_1.raan,
    norm(CHASER_1.to_rv(MU_MARS)[3:6]),
)
print(f"dv1 = {dv_i_raan:.3f} m/s")

dv_burn1 = dv_i_raan * unit(CHASER_1.h(MU_MARS))

CHASER_1A = CHASER_1.with_dv(dv_burn1, MU_MARS)

dv1 = 123.328 m/s


In [ ]:
print_diffs(TARGET_1, CHASER_1, CHASER_1A, MU_MARS)

### From 1 -> 2

In [700]:
# Very circular, so we should just depart at whichever is sooner
periapsis_is_closer = CHASER_1A.ta > np.pi
print(f"{periapsis_is_closer = }")

d_ta_chaser_1_to_2 = (
    (2 * np.pi - CHASER_1A.ta)
    if periapsis_is_closer
    else wrap_to_2pi(np.pi - CHASER_1A.ta)
)
CHASER_2_exp, dt_1_to_2 = orbit_sweep(CHASER_1A, d_ta_chaser_1_to_2, MU_MARS)

# Propagate 1 to 2
t2 = t1 + dt_1_to_2
ts_1_to_2 = np.linspace(t1, t2, 3000)
rv_c_1_to_2 = propagate_two_body(
    CHASER_1A.to_rv(MU_MARS), ts_1_to_2, mu=MU_MARS, j2=J2, r_cb=R_MARS
)
rv_t_1_to_2 = propagate_two_body(
    TARGET_1.to_rv(MU_MARS), ts_1_to_2, mu=MU_MARS, j2=J2, r_cb=R_MARS
)

# Turning into real
CHASER_2 = COES.from_rv(rv_c_1_to_2[-1], MU_MARS)
TARGET_2 = COES.from_rv(rv_t_1_to_2[-1], MU_MARS)

ts_0_to_2 = np.concatenate((ts_0_to_1[:-1], ts_1_to_2))
rv_c_0_to_2 = np.concatenate((rv_c_0_to_1[:-1], rv_c_1_to_2))
rv_t_0_to_2 = np.concatenate((rv_t_0_to_1[:-1], rv_t_1_to_2))

periapsis_is_closer = True


### Time 2-3: Hohmann burns

In [701]:
a_trans, dt_2_to_3, dv_a, dv_b = guid_coaxial_hohmann(
    CHASER_2.r_p,
    CHASER_2.r_a,
    TARGET_2.r_p,
    TARGET_2.r_a,
    depart_at_periapsis=periapsis_is_closer,
    mu=MU_MARS,
)

print(f"sma of transfer orbit: {a_trans} m")
print(f"Transfer orbit (half) period {dt_2_to_3 * SEC_TO_MIN} min")
print(f"{dv_a = :.3f} m/s (negative means retrograde)")
print(f"{dv_b = :.3f} m/s (negative means retrograde)")

sma of transfer orbit: 3682033.1821418973 m
Transfer orbit (half) period 56.528345216583524 min
dv_a = -14.871 m/s (negative means retrograde)
dv_b = 3.204 m/s (negative means retrograde)


In [702]:
# Burning chaser
dv_burn2 = dv_a * unit(CHASER_2.to_rv(MU_MARS)[3:6])
CHASER_2A = CHASER_2.with_dv(dv_burn2, MU_MARS)

print_diffs(TARGET_2, CHASER_2, CHASER_2A, MU_MARS)

t3 = t2 + dt_2_to_3
ts_2_to_3 = np.linspace(t2, t3, 3000)
rv_c_2_to_3 = propagate_two_body(
    CHASER_2A.to_rv(MU_MARS), ts_2_to_3, mu=MU_MARS, j2=J2, r_cb=R_MARS
)
rv_t_2_to_3 = propagate_two_body(
    TARGET_2.to_rv(MU_MARS), ts_2_to_3, mu=MU_MARS, j2=J2, r_cb=R_MARS
)

# Turning into real
CHASER_3 = COES.from_rv(rv_c_2_to_3[-1], MU_MARS)
TARGET_3 = COES.from_rv(rv_t_2_to_3[-1], MU_MARS)

ts_0_to_3 = np.concatenate((ts_0_to_2[:-1], ts_2_to_3))
rv_c_0_to_3 = np.concatenate((rv_c_0_to_2[:-1], rv_c_2_to_3))
rv_t_0_to_3 = np.concatenate((rv_t_0_to_2[:-1], rv_t_2_to_3))

target            : a = 3688.967 km, e = 0.00188, i = 44.945 deg, raan = 14.442 deg, aop = 199.912 deg, ta = 95.241 deg (E_sp = -5.805 km2/s2)
chaser before     : a = 3714.497 km, e = 0.00874, i = 45.037 deg, raan = 14.559 deg, aop = 348.103 deg, ta = 0.622 deg (E_sp = -5.765 km2/s2)
before - target   : a = 25.531 km, e = 0.00685, i = 0.091 deg, raan = 0.117 deg, aop = 148.191 deg, ta = -94.619 deg (E_sp = -838.754 km2/s2)

chaser after      : a = 3682.033 km, e = 0.00009, i = 45.037 deg, raan = 14.559 deg, aop = 256.352 deg, ta = 92.373 deg (E_sp = -5.816 km2/s2)
change            : a = -32.464 km, e = -0.00864, i = 0.000 deg, raan = 0.000 deg, aop = -91.751 deg, ta = 91.751 deg (E_sp = 659.620 km2/s2)
after - target    : a = -6.933 km, e = -0.00179, i = 0.091 deg, raan = 0.117 deg, aop = 56.440 deg, ta = -2.868 deg (E_sp = 3088.527 km2/s2)


In [703]:
# States at 3
# CHASER_3 = pkepler_coes(CHASER_2A, dt_2_to_3, MU_MARS) # SHOULD BE THE SAME AS ORBIT SWEEPING
# TARGET_3 = pkepler_coes(TARGET_2, dt_2_to_3, MU_MARS)

# Burning chaser
dv_burn3 = dv_b * unit(CHASER_3.to_rv(MU_MARS)[3:6])
CHASER_3A = CHASER_3.with_dv(dv_burn3, MU_MARS)

print_diffs(TARGET_3, CHASER_3, CHASER_3A, MU_MARS)

target            : a = 3688.889 km, e = 0.00125, i = 44.945 deg, raan = 14.128 deg, aop = 201.942 deg, ta = 272.580 deg (E_sp = -5.805 km2/s2)
chaser before     : a = 3682.136 km, e = 0.00321, i = 45.037 deg, raan = 14.240 deg, aop = 165.437 deg, ta = 4.068 deg (E_sp = -5.816 km2/s2)
before - target   : a = -6.753 km, e = 0.00196, i = 0.092 deg, raan = 0.112 deg, aop = -36.505 deg, ta = 91.489 deg (E_sp = 3171.089 km2/s2)

chaser after      : a = 3689.092 km, e = 0.00509, i = 45.037 deg, raan = 14.240 deg, aop = 166.934 deg, ta = 2.571 deg (E_sp = -5.805 km2/s2)
change            : a = 6.956 km, e = 0.00188, i = 0.000 deg, raan = 0.000 deg, aop = 1.497 deg, ta = -1.497 deg (E_sp = -3078.510 km2/s2)
after - target    : a = 0.203 km, e = 0.00384, i = 0.092 deg, raan = 0.112 deg, aop = -35.008 deg, ta = 89.992 deg (E_sp = -105447.336 km2/s2)


### From 3 -> 4

In [704]:
# Nothing
dt_3_to_4 = 0
t4 = t3 + dt_3_to_4

CHASER_4 = CHASER_3A.copy()
TARGET_4 = TARGET_3.copy()

ts_0_to_4 = ts_0_to_3.copy()
rv_c_0_to_4 = rv_c_0_to_3.copy()
rv_t_0_to_4 = rv_t_0_to_3.copy()

### Time 4-5: Phasing entry

In [705]:
# I'm kinda impatient
k_target = 16
k_chaser = 16


d_ta = wrap_to_pi(CHASER_4.aol - TARGET_4.aol)
dv_4_5_tot, dt_4_to_5, a_phase = guid_phasing_same_orbit(
    TARGET_4.a, d_ta, MU_MARS, k_target, k_chaser, deg=False, r_cb=R_MARS
)

dt_per_phase = dt_4_to_5 / k_chaser


print(f"Orbital separation: {d_ta * RAD_TO_DEG:.5f}º")
print(
    f"sma of transfer orbit: {a_phase * M_TO_KM} m (compared to target {TARGET_4.a * M_TO_KM:.2f} km and {CHASER_4.a * M_TO_KM:.2f} km)"
)
print(
    f"Transfer duration {dt_4_to_5 * SEC_TO_MIN} min ({dt_4_to_5 * SEC_TO_HOUR} hr, {dt_4_to_5 * SEC_TO_DAY} day))"
)
print(f"{dv_4_5_tot = :.3f} m/s (negative means retrograde)")

Orbital separation: 54.98326º
sma of transfer orbit: 3712.3268426127393 m (compared to target 3688.89 km and 3689.09 km)
Transfer duration 1831.2768783263693 min (30.52128130543949 hr, 1.271720054393312 day))
dv_4_5_tot = 21.479 m/s (negative means retrograde)


In [706]:
# Burning chaser
dv_burn4 = dv_4_5_tot / 2 * unit(CHASER_4.to_rv(MU_MARS)[3:6])
CHASER_4A = CHASER_4.with_dv(dv_burn4, MU_MARS)

print_diffs(TARGET_4, CHASER_4, CHASER_4A, MU_MARS)

# Propagate 4 to 5
t5 = t4 + dt_4_to_5
ts_4_to_5 = np.linspace(t4, t5, 3000)
rv_c_4_to_5 = propagate_two_body(
    CHASER_4A.to_rv(MU_MARS), ts_4_to_5, mu=MU_MARS, j2=J2, r_cb=R_MARS
)
rv_t_4_to_5 = propagate_two_body(
    TARGET_4.to_rv(MU_MARS), ts_4_to_5, mu=MU_MARS, j2=J2, r_cb=R_MARS
)

ts_0_to_5 = np.concatenate((ts_0_to_4[:-1], ts_4_to_5))
rv_c_0_to_5 = np.concatenate((rv_c_0_to_4[:-1], rv_c_4_to_5))
rv_t_0_to_5 = np.concatenate((rv_t_0_to_4[:-1], rv_t_4_to_5))

CHASER_5 = COES.from_rv(rv_c_4_to_5[-1], MU_MARS)
TARGET_5 = COES.from_rv(rv_t_4_to_5[-1], MU_MARS)

target            : a = 3688.889 km, e = 0.00125, i = 44.945 deg, raan = 14.128 deg, aop = 201.942 deg, ta = 272.580 deg (E_sp = -5.805 km2/s2)
chaser before     : a = 3689.092 km, e = 0.00509, i = 45.037 deg, raan = 14.240 deg, aop = 166.934 deg, ta = 2.571 deg (E_sp = -5.805 km2/s2)
before - target   : a = 0.203 km, e = 0.00384, i = 0.092 deg, raan = 0.112 deg, aop = -35.008 deg, ta = 89.992 deg (E_sp = -105447.336 km2/s2)

chaser after      : a = 3712.652 km, e = 0.01140, i = 45.037 deg, raan = 14.240 deg, aop = 168.351 deg, ta = 1.155 deg (E_sp = -5.768 km2/s2)
change            : a = 23.560 km, e = 0.00631, i = 0.000 deg, raan = 0.000 deg, aop = 1.417 deg, ta = -1.417 deg (E_sp = -908.909 km2/s2)
after - target    : a = 23.763 km, e = 0.01015, i = 0.092 deg, raan = 0.112 deg, aop = -33.592 deg, ta = 88.575 deg (E_sp = -901.141 km2/s2)


In [707]:
# Burning chaser
dv_burn5 = -dv_4_5_tot / 2 * unit(CHASER_5.to_rv(MU_MARS)[3:6])
CHASER_5A = CHASER_5.with_dv(dv_burn5, MU_MARS)  # Since burning back now

print_diffs(TARGET_5, CHASER_5, CHASER_5A, MU_MARS)

target            : a = 3696.491 km, e = 0.00307, i = 45.004 deg, raan = 3.913 deg, aop = 192.893 deg, ta = 344.096 deg (E_sp = -5.793 km2/s2)
chaser before     : a = 3712.436 km, e = 0.01127, i = 45.036 deg, raan = 4.194 deg, aop = 182.567 deg, ta = 11.064 deg (E_sp = -5.768 km2/s2)
before - target   : a = 15.945 km, e = 0.00820, i = 0.032 deg, raan = 0.281 deg, aop = -10.327 deg, ta = 26.968 deg (E_sp = -1343.029 km2/s2)

chaser after      : a = 3688.885 km, e = 0.00521, i = 45.036 deg, raan = 4.194 deg, aop = 169.277 deg, ta = 24.354 deg (E_sp = -5.805 km2/s2)
change            : a = -23.550 km, e = -0.00606, i = 0.000 deg, raan = 0.000 deg, aop = -13.290 deg, ta = 13.290 deg (E_sp = 909.286 km2/s2)
after - target    : a = -7.606 km, e = 0.00214, i = 0.032 deg, raan = 0.281 deg, aop = -23.617 deg, ta = 40.258 deg (E_sp = 2815.484 km2/s2)


In [708]:
dv_sum = sum(
    [norm(dv_burn1), norm(dv_burn2), norm(dv_burn3), norm(dv_burn4), norm(dv_burn5)]
)
print(f"dv_tot: {dv_sum}")

dv_tot: 162.8815943174682


### Onward

In [709]:
dt_5_to_6 = 100 * DAY_TO_SEC
t6 = t5 + dt_5_to_6
ts_5_to_6 = np.linspace(t5, t6, 3000)
# ts_5_to_6 = np.linspace(t5, t6, 30_000) # will take like a minute to run (GOOD FOR VERIFYING THAT IT DOESN'T DRIFT)
rv_c_5_to_6 = propagate_two_body(
    CHASER_5A.to_rv(MU_MARS), ts_5_to_6, mu=MU_MARS, j2=J2, r_cb=R_MARS
)
rv_t_5_to_6 = propagate_two_body(
    TARGET_5.to_rv(MU_MARS), ts_5_to_6, mu=MU_MARS, j2=J2, r_cb=R_MARS
)

ts_0_to_6 = np.concatenate((ts_0_to_5[:-1], ts_5_to_6))
rv_c_0_to_6 = np.concatenate((rv_c_0_to_5[:-1], rv_c_5_to_6))
rv_t_0_to_6 = np.concatenate((rv_t_0_to_5[:-1], rv_t_5_to_6))

In [ ]:
burns = [
    (t1, "Burn 1 (plane)"),
    (t2, "Burn 2 (Hohmann)"),
    (t3, "Burn 3 (circularize)"),
    (t4, "Burn 4 (Phase)"),
    (t5, "Burn 5 (circularize)"),
]

fig_sep = plot_separation(ts_0_to_5, rv_c_0_to_5, rv_t_0_to_5, burns)

In [711]:
# Chaser relative to the target, in the target's rotating RSW frame [m, m/s]
rv_c_5_to_6_rsw = np.array(
    [
        hcw_pci_to_rsw(rv_t, rv_c)
        for rv_t, rv_c in tqdm(zip(rv_t_5_to_6, rv_c_5_to_6), total=len(ts_5_to_6))
    ]
)

100%|██████████| 3000/3000 [00:00<00:00, 21108.02it/s]


In [ ]:
r_rsw = rv_c_5_to_6_rsw[:, 0:3]
print(
    f"relative position at t5: R = {r_rsw[0, 0]:.3f}, S = {r_rsw[0, 1]:.3f}, W = {r_rsw[0, 2]:.3f}"
)
print(f"max |relative position| over 5 -> 6: {norm(r_rsw, axis=1).max():.3f}")

fig_sep = plot_separation(ts_5_to_6, r_rsw, np.zeros_like(r_rsw))

In [713]:
# HCW view of the target's RSW frame: along-track (S) horizontal with +S to the left, radial (R) up
s_km = r_rsw[:, 1] * M_TO_KM
r_km = r_rsw[:, 0] * M_TO_KM
t_rel_hr = (ts_5_to_6 - t5) * SEC_TO_HOUR

fig_hcw = go.Figure()
fig_hcw.add_trace(
    go.Scattergl(  # WebGL handles the full-resolution trajectory
        x=s_km,
        y=r_km,
        mode="lines",
        name="Chaser",
        line={"color": "#2a78d6", "width": 2},
        customdata=t_rel_hr,
        hovertemplate="S = %{x:.3f} km<br>R = %{y:.3f} km<br>t = %{customdata:.2f} hr<extra></extra>",
    )
)
fig_hcw.add_trace(
    go.Scatter(
        x=[s_km[0]],
        y=[r_km[0]],
        mode="markers+text",
        name="Start (burn 5)",
        marker={
            "color": "#2a78d6",
            "size": 9,
            "line": {"color": "#fcfcfb", "width": 2},
        },
        text=["start"],
        textposition="top center",
        textfont={"color": "#52514e"},
        hovertemplate="start<br>S = %{x:.3f} km<br>R = %{y:.3f} km<extra></extra>",
    )
)
fig_hcw.add_trace(
    go.Scatter(
        x=[0.0],
        y=[0.0],
        mode="markers+text",
        name="Target",
        marker={"color": "#0b0b0b", "size": 10, "symbol": "x"},
        text=["target"],
        textposition="bottom center",
        textfont={"color": "#52514e"},
        hovertemplate="target<extra></extra>",
    )
)
fig_hcw.update_xaxes(
    title="Along-track S [km]  (+V-bar, direction of motion)",
    autorange="reversed",
    zeroline=True,
    zerolinecolor="#c3c2b7",
)
fig_hcw.update_yaxes(
    title="Radial R [km]  (+R-bar, away from Mars)",
    scaleanchor="x",  # equal km per km on both axes
    zeroline=True,
    zerolinecolor="#c3c2b7",
)
fig_hcw.update_layout(
    title="Chaser relative to target, in-plane (HCW view)",
    template="plotly_white",
    showlegend=False,
    height=550,
)
fig_hcw.show()

## Visualization

In [722]:
vectors = [
    {
        "direction": n_hat,
        "length": 20000,
        "name": "Node line",
        "color": "goldenrod",
        "label": True,
    },
]

points = [
    # {
    #     "name": "r_t1",
    #     "location": rv_t_0_to_1[-1,0:3] * M_TO_KM,
    #     "color": "blue",
    # },
    # {
    #     "name": "r_c1",
    #     "location": rv_c_0_to_1[-1,0:3] * M_TO_KM,
    #     "color": "yellow",
    # },
]

fig = plot_3d(
    [rv_t_0_to_1[:, 0:3] * M_TO_KM, rv_c_0_to_1[:, 0:3] * M_TO_KM],
    [ts_0_to_1, ts_0_to_1],
    labels=["Target", "Chaser"],
    trace_colors=["purple", "green"],
    bodies={"name": "Mars", "radius": R_MARS * M_TO_KM, "color": "orangered"},
    title=f"Target and chaser, {N_ORBITS} orbits (MCI)",
    vectors=vectors,
    points=points,
    units="km",
)